# CS-781 Capstone, Phase 4: Court Judgment Prediction (CJPE) with the Phase-1 Mamba-2 / attention hybrid (111M)

One notebook, four presets (three main experiments + one optional extra), each producing its own `submission_<EXP>.zip`. Set `EXPERIMENT` in section 1, run all cells, repeat for the next preset.

**Task:** predict the binary outcome (0/1) of a court case from its text. Metric: macro-F1 (both classes matter).

**Long documents.** Judgments are far longer than the model context (1280 tokens), so every case is handled hierarchically:
1. The text is normalised exactly like Phase 1 and tokenised **lazily** (only the cases actually used).
2. It is cut into 1024-token chunks (aligned from the end of the document) and at most `MAX_CHUNKS` chunks are kept, chosen by `CHUNK_POLICY`: `first`, `last`, `head_tail` (default: start and end of the judgment), or `uniform`.
3. The backbone encodes each chunk; a chunk vector = mean of token states + the last token's state, plus a learned embedding of the chunk's relative position in the case.
4. **Attention pooling** over the chunk vectors gives one case vector (`AGG="gru"`/`"mean"` are available), then a small classifier gives P(outcome = 1).
5. The decision threshold is tuned on the dev split for macro-F1 (`USE_TUNED_THRESHOLD`).

**Compute control.** The training split may contain many thousands of long cases. A benchmark cell measures throughput and, if one pass over all cases would exceed `TARGET_TRAIN_MINUTES`, samples that many cases **per epoch** (a fresh random subset each epoch), so the learning-rate schedule still completes inside your budget.

| Preset | Technique | Trainable |
|---|---|---|
| `M1_adapter` | Bottleneck adapters (Pfeiffer-style, one per layer) | adapters + head |
| `M2_partial` | Partial fine-tuning (top 5 of 20 layers) | top layers + head |
| `M3_kd` | Knowledge distillation into a half-depth student | whole student |
| `M4_lorma` | (extra) **LORMA** multiplicative low-rank | LORMA matrices + head |

**BitFit on a bias-free network.** The Phase-1 models have no bias terms (every linear layer is `bias=False`), so classic BitFit has nothing to tune. The implementation adds a zero-initialised bias vector to every linear layer and trains only those (plus the network's existing bias-like parameters, e.g. the Mamba convolution bias and `dt_bias`). Describe it as "BitFit adapted to a bias-free architecture" in the report.

**About LORMA (please read).** The assignment cites LORMA only as reference [12] and does not define it. The implementation is my reconstruction from memory of the idea of *multiplicative* low-rank adaptation: `W' = (I + s·B·A)·W0` applied to the layer output, or `W0·(I + s·B·A)` applied to the layer input (`side="auto"` picks the cheaper side), with `B` zero-initialised so training starts from the Phase-1 function. I could not check the paper, so **compare `LoRMALinear` with reference [12] and edit it if the formulation differs** before describing it as a faithful reproduction.

**Knowledge distillation** works on case-level probabilities, so any run of any of the three Phase-4 notebooks can teach any student, regardless of tokenizer or architecture. Every non-KD run saves a `teacher_<EXP>.pkl` (probabilities for a subset of training cases, all dev and all test cases). The student trains on the training cases that have teacher outputs.

## 1. Configuration

In [54]:
import os, sys, json, re, math, time, random, zipfile, heapq, shutil, subprocess, hashlib, pickle, glob
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

DEBUG = False                 # True = tiny synthetic smoke test
SEED = 42
EXPERIMENT = "M4_lorma"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# ---- Phase-1 assets: Mamba-hybrid checkpoint and ITS OWN bpe_merges.json (137M-token-corpus tokenizer) ----
CKPT_PATH = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/best.pt"
TOK_PATH  = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/bpe_merges.json"
OUTPUT_DIR = "/kaggle/working/phase4_cjpe"
EOS_PLACEHOLDER = "<|endoftext|>"
TEACHER_PKL = "/kaggle/working/phase4_cjpe/M1_adapter/teacher_M1_adapter.pkl"      # for the *_kd presets: <OUTPUT_DIR>/<EXP>/teacher_<EXP>.pkl of an earlier run

PRESETS = {
    "M1_adapter": dict(method="adapter", lr_backbone=1e-3, lr_head=1e-3, adapter_dim=64,
                       desc="Bottleneck adapters after every layer (backbone frozen)"),
    "M2_partial": dict(method="partial", lr_backbone=1e-4, lr_head=1e-3, train_top=5,
                       desc="Partial fine-tuning: only the top 5 of 20 layers (+ final norm) and the head are trained"),
    "M3_kd":      dict(method="full",    lr_backbone=1e-4, lr_head=1e-3, student_keep="odd", kd_alpha=0.7, kd_temp=2.0,
                       desc="Knowledge distillation: half-depth student (odd layers) learns from teacher document probabilities"),
    "M4_lorma":   dict(method="lorma",   lr_backbone=1e-3, lr_head=1e-3, r=16, alpha=32, dropout=0.05, side="auto",
                       desc="(optional extra) LORMA multiplicative low-rank update on every linear layer"),
}

# ---- document representation (shared by all presets) ----
CHUNK_TOK, MAX_CHUNKS = 1024, 4        # tokens per chunk (<= model context) and chunks kept per case
CHUNK_POLICY = "head_tail"             # "head_tail" | "last" | "first" | "uniform"
AGG = "attn"                           # chunk aggregation: "attn" | "gru" | "mean"
HEAD_DIM, HEAD_DROPOUT = 256, 0.2
# ---- training budget (shared) ----
DOCS_PER_MICRO, GRAD_ACCUM = 1, 16     # cases per micro-batch (each case = up to MAX_CHUNKS sequences) and accumulation
EPOCHS = 3
TRAIN_DOCS_PER_EPOCH = None            # None = every training case each epoch (the benchmark cell may set this)
AUTO_FIT_BUDGET, TARGET_TRAIN_MINUTES = True, 90
MAX_DEV_DOCS = None                    # cap on dev cases used for epoch selection (None = all)
WARMUP_FRAC, WEIGHT_DECAY, CLIP = 0.06, 0.01, 1.0
CLASS_WEIGHT = "balanced"              # "balanced" | "none"
DEDUP_EXACT = True                     # drop exact duplicates inside train (same text and label)
DROP_TRAIN_DEV_OVERLAP = True          # drop train cases whose text also appears in dev (keeps the dev score honest)
USE_TUNED_THRESHOLD = True             # tune the decision threshold on dev for macro-F1
FINAL_RETRAIN = False                  # True = retrain on train+dev for the best epoch count (doubles the cost)
GRAD_CKPT = False                      # True = less memory, ~30% slower
INFER_DOCS = 2                         # cases per inference batch
PROBE_MAX_DOCS = 300                   # dev cases used by the section probe
TEACHER_SAVE_TRAIN_DOCS = 4000         # training cases whose probabilities are saved for later distillation
SAVE_TEACHER = True
ALLOW_LARGER_STUDENT = False

if DEBUG:
    CHUNK_TOK, MAX_CHUNKS, HEAD_DIM = 96, 3, 32
    GRAD_ACCUM, EPOCHS, INFER_DOCS, PROBE_MAX_DOCS, TEACHER_SAVE_TRAIN_DOCS = 2, 3, 4, 20, 40
    TARGET_TRAIN_MINUTES = 5

EXP = PRESETS[EXPERIMENT]
EXP_DIR = f"{OUTPUT_DIR}/{EXPERIMENT}"
os.makedirs(EXP_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "", "| experiment:", EXPERIMENT, "-", EXP["desc"])

device: cuda | Tesla T4 | experiment: M4_lorma - (optional extra) LORMA multiplicative low-rank update on every linear layer


## 2. Phase-1 code, reused unchanged
`normalize_text`, the from-scratch `ByteBPE` tokenizer and the model architecture, copied from the Phase-1 notebooks so the checkpoint loads exactly.

In [55]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [56]:
@dataclass
class HybridConfig:
    vocab_size: int
    n_ctx: int = 1280
    d_model: int = 768
    pattern: str = "MMMA" * 6        # M = Mamba-2 (SSD) layer, A = attention + SwiGLU layer
    # --- Mamba-2 / SSD layer ---
    expand: int = 2                  # d_inner = expand * d_model
    headdim: int = 64                # SSM head size P  (n_ssm_heads = d_inner / headdim)
    d_state: int = 64                # state size N per head
    d_conv: int = 4                  # causal depthwise conv width
    chunk: int = 64                  # SSD chunk length Q
    # --- attention layer ---
    n_head: int = 12
    n_kv_head: int = 4
    n_ffn: int = 2048
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class GatedRMSNorm(nn.Module):
    """Mamba-2 output norm: RMSNorm(y * silu(z))."""
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, y, z):
        y = y * F.silu(z)
        yf = y.float()
        return (yf * torch.rsqrt(yf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(y) * self.weight

# ------------------------------------------------------------------------------------------
# State-space duality (SSD) scan, pure PyTorch. Computes, for every head h,
#     s_t = exp(dt_t * A_h) * s_{t-1} + dt_t * (x_t outer B_t)      y_t = C_t . s_t
# in chunks of Q tokens: inside a chunk it is a masked "attention-like" matmul; across chunks only a
# small recurrence on the (P x N) states is needed. Decay math is fp32, big matmuls follow autocast.
# ------------------------------------------------------------------------------------------
def ssd_chunked(X, dt, A, Bm, Cm, Q):
    """X (B,T,H,P) | dt (B,T,H) fp32 | A (H,) fp32 <0 | Bm,Cm (B,T,N). Returns Y (B,T,H,P), final state (B,H,P,N) fp32."""
    Bsz, T, H, P = X.shape
    N = Bm.size(-1)
    pad = (-T) % Q
    if pad:                                                   # dt = 0 on padding => state is untouched
        X = F.pad(X, (0, 0, 0, 0, 0, pad)); dt = F.pad(dt, (0, 0, 0, pad))
        Bm = F.pad(Bm, (0, 0, 0, pad)); Cm = F.pad(Cm, (0, 0, 0, pad))
    Tp = T + pad; nC = Tp // Q
    Xc = (X * dt.to(X.dtype)[..., None]).view(Bsz, nC, Q, H, P)             # dt-scaled input
    Bc, Cc = Bm.reshape(Bsz, nC, Q, N), Cm.reshape(Bsz, nC, Q, N)
    a_cum = (dt * A).view(Bsz, nC, Q, H).permute(0, 3, 1, 2).cumsum(-1)     # (B,H,nC,Q) fp32, non-increasing

    # 1) inside each chunk: Y = (C B^T o L) X, with L[i,j] = exp(a_i - a_j) for i >= j
    tril = torch.tril(torch.ones(Q, Q, dtype=torch.bool, device=X.device))
    L = torch.exp((a_cum[..., :, None] - a_cum[..., None, :]).masked_fill(~tril, float("-inf")))
    G = torch.einsum("bcqn,bcsn->bcqs", Cc, Bc)                              # (B,nC,Q,Q)
    M = (G[:, None].float() * L).to(Xc.dtype)                                # (B,H,nC,Q,Q)
    Y_diag = torch.einsum("bhcqs,bcshp->bcqhp", M, Xc)

    # 2) the state each chunk would produce on its own
    decay_states = torch.exp(a_cum[..., -1:] - a_cum)                        # (B,H,nC,Q)
    Xw = Xc * decay_states.permute(0, 2, 3, 1)[..., None].to(Xc.dtype)
    states = torch.einsum("bcshp,bcsn->bchpn", Xw, Bc)                       # (B,nC,H,P,N)

    # 3) pass states across chunks (tiny recurrence, fp32)
    with torch.autocast(device_type=X.device.type, enabled=False):
        states = torch.cat([torch.zeros_like(states[:, :1]), states.float()], 1)     # (B,nC+1,H,P,N)
        cs = F.pad(a_cum[..., -1], (1, 0)).cumsum(-1)                                # (B,H,nC+1)
        seg = cs[..., :, None] - cs[..., None, :]
        keep = torch.tril(torch.ones(nC + 1, nC + 1, dtype=torch.bool, device=X.device))
        decay_chunk = torch.exp(seg.masked_fill(~keep, float("-inf")))
        new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)
    prev_states, final_state = new_states[:, :-1], new_states[:, -1]

    # 4) contribution of everything before the chunk to each token inside it
    out_decay = torch.exp(a_cum).permute(0, 2, 3, 1)[..., None].to(Xc.dtype)         # (B,nC,Q,H,1)
    Y_off = torch.einsum("bcqn,bchpn->bcqhp", Cc, prev_states.to(Cc.dtype)) * out_decay
    return (Y_diag + Y_off).reshape(Bsz, Tp, H, P)[:, :T], final_state

class Mamba2Block(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.d_inner, self.P, self.N, self.K, self.Q = c.expand * c.d_model, c.headdim, c.d_state, c.d_conv, c.chunk
        assert self.d_inner % self.P == 0
        self.H = self.d_inner // self.P
        self.conv_dim = self.d_inner + 2 * self.N
        self.in_proj = nn.Linear(c.d_model, 2 * self.d_inner + 2 * self.N + self.H, bias=False)   # z | x B C | dt
        self.conv1d = nn.Conv1d(self.conv_dim, self.conv_dim, self.K, groups=self.conv_dim, padding=self.K - 1)
        dt = torch.exp(torch.rand(self.H) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))            # softplus^-1(dt) with dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(self.H).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.H))
        self.norm = GatedRMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, c.d_model, bias=False)

    def forward(self, x, valid=None, want_state=False):
        """Parallel (chunked) form for training and prompt prefill. valid: (B,T) bool, False = left-padding."""
        B, T, _ = x.shape
        z, xBC, dt = self.in_proj(x).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        if valid is not None:
            xBC = xBC * valid[..., None].to(xBC.dtype)              # padding looks like the zero boundary of the conv
        xBC_t = xBC.transpose(1, 2)
        xBC = F.silu(self.conv1d(xBC_t)[..., :T]).transpose(1, 2)
        xs, Bm, Cm = xBC.split([self.d_inner, self.N, self.N], dim=-1)
        dtf = F.softplus(dt.float() + self.dt_bias)
        if valid is not None:
            dtf = dtf * valid[..., None].float()                    # dt = 0  =>  state passes through padding unchanged
        A = -torch.exp(self.A_log.float())
        X = xs.reshape(B, T, self.H, self.P)
        Y, state = ssd_chunked(X, dtf, A, Bm, Cm, self.Q)
        Y = Y + X * self.D.to(X.dtype)[None, None, :, None]
        out = self.out_proj(self.norm(Y.reshape(B, T, self.d_inner), z))
        if not want_state:
            return out, None
        conv_state = xBC_t[..., -(self.K - 1):]
        if conv_state.size(-1) < self.K - 1:
            conv_state = F.pad(conv_state, (self.K - 1 - conv_state.size(-1), 0))
        return out, (conv_state, state)

    def step(self, x, st):
        """Recurrent form for one token (B,1,D): constant memory, constant time."""
        conv_state, ssm = st
        z, xBC, dt = self.in_proj(x[:, 0]).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        window = torch.cat([conv_state, xBC[..., None]], dim=-1)                    # (B, conv_dim, K)
        conv = (window.float() * self.conv1d.weight[:, 0, :].float()).sum(-1) + self.conv1d.bias.float()
        xs, Bm, Cm = F.silu(conv).split([self.d_inner, self.N, self.N], dim=-1)
        X = xs.view(-1, self.H, self.P)
        dtf = F.softplus(dt.float() + self.dt_bias)                                 # (B,H)
        dA = torch.exp(dtf * -torch.exp(self.A_log.float()))
        ssm = ssm * dA[..., None, None] + dtf[..., None, None] * X[..., None] * Bm[:, None, None, :]
        y = (ssm * Cm[:, None, None, :]).sum(-1) + self.D[None, :, None] * X        # (B,H,P)
        out = self.out_proj(self.norm(y.reshape(-1, 1, self.d_inner).to(z.dtype), z[:, None]))
        return out, (window[..., 1:], ssm)

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (*, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    """Grouped-query attention with RoPE (only 1 in 4 layers is attention in the default pattern)."""
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.nh, self.nkv, self.hd = c.n_head, c.n_kv_head, c.d_model // c.n_head
        self.q = nn.Linear(c.d_model, self.nh * self.hd, bias=False)
        self.k = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.v = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.o = nn.Linear(self.nh * self.hd, c.d_model, bias=False)
    def forward(self, x, cos, sin, mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        new_cache = None
        if cache is not None:
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            new_cache = (k, v)
        rep = self.nh // self.nkv
        if rep > 1:
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=(mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.gate = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.up = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class MambaLayer(nn.Module):
    kind = "M"
    def __init__(self, c):
        super().__init__()
        self.norm, self.mixer = RMSNorm(c.d_model), Mamba2Block(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, st = self.mixer(self.norm(x), valid, want_cache)
        return x + h, st
    def step(self, x, cos, sin, key_mask, cache):
        h, st = self.mixer.step(self.norm(x), cache)
        return x + h, st

class AttnLayer(nn.Module):
    kind = "A"
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.d_model), Attention(c)
        self.n2, self.mlp = RMSNorm(c.d_model), SwiGLU(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, cache = self.attn(self.n1(x), cos, sin, mask, (None, None) if want_cache else None)
        x = x + h
        return x + self.mlp(self.n2(x)), cache
    def step(self, x, cos, sin, key_mask, cache):
        h, cache = self.attn(self.n1(x), cos, sin, key_mask, cache)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalHybrid(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        assert set(c.pattern) <= {"M", "A"}
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([MambaLayer(c) if ch == "M" else AttnLayer(c) for ch in c.pattern])
        self.norm = RMSNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                                   # weight tying
        cos, sin = build_rope(c.d_model // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                                     # scaled residual init
            if n.endswith("out_proj.weight") or n.endswith("attn.o.weight") or n.endswith("mlp.down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(len(c.pattern)))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, valid=None, want_cache=False, last_only=False):
        """Training: model(x, y). Prompt prefill: model(idx, valid=left_pad_mask, want_cache=True, last_only=True)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        mask = None
        if valid is None:
            cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        else:                                                                    # left-padded batch
            pos = (valid.long().cumsum(1) - 1).clamp(min=0)
            cos, sin = self.cos[pos][:, None], self.sin[pos][:, None]
            causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
            eye = torch.eye(T, dtype=torch.bool, device=idx.device)              # keeps pad rows finite
            mask = (causal[None] & (valid[:, None, :] | eye[None]))[:, None]
        caches = [] if want_cache else None
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, c_, s_, b=blk: b(x_, c_, s_, None, None, False)[0], x, cos, sin, use_reentrant=False)
            else:
                x, c_new = blk(x, cos, sin, mask, valid, want_cache)
                if want_cache: caches.append(c_new)
        x = self.norm(x)
        if last_only: x = x[:, -1:]
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, caches

    def step(self, tok, pos, key_mask, caches):
        """One decoding step. tok (B,1); pos (B,) = position of tok; key_mask (B,1,1,L_cached+1) bool."""
        x = self.tok_emb(tok)
        cos, sin = self.cos[pos][:, None, None, :], self.sin[pos][:, None, None, :]
        new = []
        for blk, cache in zip(self.blocks, caches):
            x, c_new = blk.step(x, cos, sin, key_mask, cache)
            new.append(c_new)
        return self.head(self.norm(x)), new

## 3. Load the Phase-1 checkpoint and tokenizer

In [57]:
for p in (CKPT_PATH, TOK_PATH):
    if not os.path.exists(p):
        print("MISSING:", p, "\nFiles under /kaggle/input:")
        for root, _, files in os.walk("/kaggle/input"):
            for f in files: print("  ", os.path.join(root, f))
        raise FileNotFoundError(p)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
p1_cfg = HybridConfig(**ck["cfg"])
tokenizer = ByteBPE.load(TOK_PATH)
EOS_ID = tokenizer.eos_id
assert p1_cfg.vocab_size == tokenizer.vocab_size, \
    f"tokenizer/checkpoint mismatch: model vocab {p1_cfg.vocab_size} vs tokenizer {tokenizer.vocab_size}. Use the bpe_merges.json this checkpoint was trained with."
print(f"checkpoint step {ck.get('step')} | vocab {p1_cfg.vocab_size} | n_ctx {p1_cfg.n_ctx} | config: {ck['cfg']}")
print("tokenizer merges md5:", hashlib.md5(open(TOK_PATH, "rb").read()).hexdigest()[:12])
assert CHUNK_TOK <= p1_cfg.n_ctx, f"CHUNK_TOK {CHUNK_TOK} exceeds the model context {p1_cfg.n_ctx}"

def build_backbone():
    m = LegalHybrid(p1_cfg)
    m.load_state_dict(ck["model"])
    return m

def backbone_dim(m): return m.c.d_model
def embed(m): return m.tok_emb
def layers(m): return m.blocks
def final_norm(m): return m.norm

def run_blocks(m, x, mask, post=None):
    T = x.size(1)
    cos, sin = m.cos[:T][None, None], m.sin[:T][None, None]
    for i, blk in enumerate(m.blocks):
        if GRAD_CKPT and m.training:
            x = torch.utils.checkpoint.checkpoint(lambda x_, b=blk: b(x_, cos, sin, None, None, False)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, cos, sin, None, None, False)
        if post is not None: x = x + post[i](x)
    return m.norm(x)

def prune_layers(m, keep):
    """Keep only the listed layers: a half-depth student that still starts from Phase-1 weights."""
    m.blocks = nn.ModuleList([m.blocks[i] for i in keep])
    return m

_probe = build_backbone().eval()
print(f"Phase-1 backbone used here: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters, {len(layers(_probe))} layers (loaded OK)")
del _probe

checkpoint step 1000 | vocab 32000 | n_ctx 1280 | config: {'vocab_size': 32000, 'n_ctx': 1280, 'd_model': 768, 'pattern': 'MMMAMMMAMMMAMMMAMMMA', 'expand': 2, 'headdim': 64, 'd_state': 64, 'd_conv': 4, 'chunk': 64, 'n_head': 12, 'n_kv_head': 4, 'n_ffn': 2048, 'rope_base': 10000.0}
tokenizer merges md5: 16ee00bde3c5
Phase-1 backbone used here: 111.0M parameters, 20 layers (loaded OK)


## 4. Phase-4 data: load, parse robustly, audit
The loader accepts a list of records or a dict keyed by id; the label may be called `label`, `labels`, `outcome`, `judgment` or `pred` and may be an int, bool or string. Output ids keep the **original type** from `test.json`.

In [58]:
from huggingface_hub import HfApi, hf_hub_download, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("capstone-phase1-token")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
_repo_files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)

def fetch(*must_contain):
    hits = [f for f in _repo_files if all(s.lower() in f.lower() for s in must_contain)]
    if not hits:
        raise FileNotFoundError(f"{must_contain} not found. Repo has: {_repo_files[:60]}")
    return hf_hub_download(HF_DATASET_REPO, hits[0], repo_type="dataset", token=HF_TOKEN)
print([f for f in _repo_files if "phase_4" in f.lower()])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


['Phase_4_CJPE/Test/test.json', 'Phase_4_CJPE/Train/dev.json', 'Phase_4_CJPE/Train/train.json']


In [59]:
P4 = {"train": fetch("phase_4", "train.json"), "dev": fetch("phase_4", "dev.json"), "test": fetch("phase_4", "test.json")}

def load_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)

def to_label(v):
    if isinstance(v, str): v = v.strip().lower(); v = {"true": 1, "false": 0, "yes": 1, "no": 0}.get(v, v)
    return int(v)

def parse_cases(obj, labelled):
    if isinstance(obj, dict) and len(obj) == 1 and isinstance(next(iter(obj.values())), list): obj = next(iter(obj.values()))
    if isinstance(obj, dict): items = [(k, dict(v) if isinstance(v, dict) else {"text": v}) for k, v in obj.items()]
    else: items = [(r.get("id", r.get("case_id", r.get("doc_id", i))), r) for i, r in enumerate(obj)]
    docs, seen = [], Counter()
    for cid, r in items:
        y = None
        if labelled:
            raw = next((r[k] for k in ("label", "labels", "outcome", "judgment", "pred", "y") if k in r), None)
            assert raw is not None, f"case {cid}: no label field among {list(r)}"
            y = to_label(raw); assert y in (0, 1), f"case {cid}: label {raw!r} is not 0/1"
        base = str(cid); seen[base] += 1
        key = base if seen[base] == 1 else f"{base}#{seen[base]}"      # internal keys stay unique even if ids repeat
        docs.append(dict(key=key, id=cid, text=str(r.get("text", r.get("case_text", ""))), y=y, ids=None))
    return docs

RAW = {"train": parse_cases(load_json(P4["train"]), True), "dev": parse_cases(load_json(P4["dev"]), True), "test": parse_cases(load_json(P4["test"]), False)}
for sp in RAW:
    for d in RAW[sp]: d["h"] = hashlib.md5(d["text"].encode("utf-8", "ignore")).hexdigest()

# ---- duplicate diagnostics (train.json merges the single- and multi-instance data, so ids can repeat) ----
for sp, ds in RAW.items():
    groups = defaultdict(list)
    for d in ds: groups[str(d["id"])].append(d)
    dup = {k: v for k, v in groups.items() if len(v) > 1}
    if dup:
        same = sum(1 for v in dup.values() if len({(x["h"], x["y"]) for x in v}) == 1)
        conflict = sum(1 for v in dup.values() if len({x["h"] for x in v}) == 1 and len({x["y"] for x in v}) > 1)
        distinct = sum(1 for v in dup.values() if len({x["h"] for x in v}) > 1)
        print(f"{sp}: {len(dup)} ids occur more than once ({sum(len(v) for v in dup.values())} cases) -> exact copies: {same} | same text, conflicting labels: {conflict} | DIFFERENT cases sharing an id: {distinct}")
assert len({str(d["id"]) for d in RAW["test"]}) == len(RAW["test"]), \
    "duplicate ids in test.json: the submission needs exactly one prediction per test id, so check with the organisers how repeated ids must be handled"

dev_hashes = {d["h"] for d in RAW["dev"]}; seen_h = set(); clean = []; removed = Counter(); n0 = len(RAW["train"])
for d in RAW["train"]:
    if DROP_TRAIN_DEV_OVERLAP and d["h"] in dev_hashes: removed["same text as a dev case"] += 1; continue
    if DEDUP_EXACT and (d["h"], d["y"]) in seen_h: removed["exact duplicate inside train"] += 1; continue
    seen_h.add((d["h"], d["y"])); clean.append(d)
RAW["train"] = clean
_lab = defaultdict(set)
for d in RAW["train"]: _lab[d["h"]].add(d["y"])
print(f"texts that occur in train with CONFLICTING labels (kept as they are; label noise): {sum(len(v) > 1 for v in _lab.values())}")
print(f"train cleaning: {n0} -> {len(clean)} cases (removed: {dict(removed) if removed else 'nothing'})")
test_h = {d["h"] for d in RAW["test"]}
print(f"overlap check: train cases whose text also appears in test: {sum(d['h'] in test_h for d in RAW['train'])} | dev cases in test: {sum(d['h'] in test_h for d in RAW['dev'])}")
print("split | cases | label balance (share of class 1) | median chars | max chars")
for sp, ds in RAW.items():
    ln = [len(d["text"]) for d in ds]; yb = f"{np.mean([d['y'] for d in ds]):.3f}" if ds[0]["y"] is not None else "-"
    print(f"{sp:5s} | {len(ds):6d} | {yb:>32s} | {int(np.median(ln)):12d} | {max(ln)}")
print("test id type:", type(RAW["test"][0]["id"]).__name__, "| sample id:", RAW["test"][0]["id"])
rs = random.Random(0); sample = rs.sample(RAW["train"], min(150, len(RAW["train"])))
tk = [len(tokenizer.encode(normalize_text(d["text"], False))) for d in sample]
print(f"tokens per case (sample of {len(sample)}): median {int(np.median(tk))}, p90 {int(np.percentile(tk, 90))}, max {max(tk)} | chunks of {CHUNK_TOK}: median {math.ceil(np.median(tk) / CHUNK_TOK)}, "
      f"cases that exceed {MAX_CHUNKS} chunks: {np.mean([t > MAX_CHUNKS * CHUNK_TOK for t in tk]):.0%}")

train: 5082 ids occur more than once (10164 cases) -> exact copies: 5082 | same text, conflicting labels: 0 | DIFFERENT cases sharing an id: 0
dev: 994 ids occur more than once (1988 cases) -> exact copies: 994 | same text, conflicting labels: 0 | DIFFERENT cases sharing an id: 0
texts that occur in train with CONFLICTING labels (kept as they are; label noise): 0
train cleaning: 37387 -> 31677 cases (removed: {'exact duplicate inside train': 5710})
overlap check: train cases whose text also appears in test: 0 | dev cases in test: 0
split | cases | label balance (share of class 1) | median chars | max chars
train |  31677 |                            0.414 |        12092 | 2492701
dev   |   1988 |                            0.500 |        16700 | 260198
test  |   1517 |                                - |        17320 | 233000
test id type: str | sample id: 1986_397
tokens per case (sample of 150): median 2475, p90 8127, max 25021 | chunks of 1024: median 3, cases that exceed 4 chunks: 3

## 5. Chunking: which parts of a judgment the model sees
Chunk boundaries are aligned from the **end** of the document (so the final chunk is always full). Policies: `first` (opening chunks), `last` (closing chunks), `head_tail` (half from each end: facts and procedural history, plus reasoning and decision), `uniform` (evenly spaced across the case). A case with at most `MAX_CHUNKS` chunks is always used whole.

In [60]:
def get_ids(d):
    if d["ids"] is None:                                   # lazy: only cases that are actually used get tokenised
        ids = tokenizer.encode(normalize_text(d["text"], False))
        d["ids"] = np.array(ids if ids else [EOS_ID], dtype=np.int32)
    return d["ids"]

def select_chunks(n, policy, M):
    """-> (list of (start, end, relative_position), total_chunks). Chunks are aligned from the end of the document."""
    k = max(1, math.ceil(n / CHUNK_TOK))
    bounds = [(max(0, n - (k - j) * CHUNK_TOK), n - (k - j - 1) * CHUNK_TOK) for j in range(k)]
    if k <= M: idx = list(range(k))
    elif policy == "first": idx = list(range(M))
    elif policy == "last": idx = list(range(k - M, k))
    elif policy == "head_tail": h = (M + 1) // 2; idx = list(range(h)) + list(range(k - (M - h), k))
    elif policy == "uniform": idx = sorted(set(np.linspace(0, k - 1, M).round().astype(int).tolist()))
    else: raise ValueError(policy)
    return [(*bounds[j], j / max(k - 1, 1)) for j in idx], k

def collate(docs, policy=None, M=None):
    policy, M = policy or CHUNK_POLICY, M or MAX_CHUNKS
    seqs, owner, rank, bucket = [], [], [], []
    for di, d in enumerate(docs):
        ids = get_ids(d); sel, d["_k"] = select_chunks(len(ids), policy, M)
        for r, (st, en, rel) in enumerate(sel):
            seqs.append(ids[st:en]); owner.append(di); rank.append(r); bucket.append(min(int(rel * 8), 7))
    N, T = len(seqs), max(len(s) for s in seqs)
    ids_t = torch.full((N, T), EOS_ID, dtype=torch.long); lens = torch.tensor([len(s) for s in seqs])
    for i, s in enumerate(seqs): ids_t[i, :len(s)] = torch.from_numpy(s.astype(np.int64))
    nch = torch.bincount(torch.tensor(owner), minlength=len(docs))
    y = torch.tensor([d["y"] if d["y"] is not None else -100 for d in docs])
    teach = torch.zeros(len(docs), 2)
    for i, d in enumerate(docs):
        if d.get("teacher") is not None: teach[i] = torch.from_numpy(d["teacher"])
    return dict(ids=ids_t.to(device), tok_mask=(torch.arange(T)[None] < lens[:, None]).to(device), lens=lens.to(device),
                owner=torch.tensor(owner).to(device), rank=torch.tensor(rank).to(device), bucket=torch.tensor(bucket).to(device),
                nch=nch.to(device), nch_cpu=nch, y=y.to(device), teach=teach.to(device))

# quick demonstration of the policies on a long synthetic length
for pol in ("first", "last", "head_tail", "uniform"):
    sel, k = select_chunks(9 * CHUNK_TOK - 17, pol, MAX_CHUNKS)
    print(f"{pol:9s}: case of 9 chunks keeps chunk indices {[round(s[2] * (k - 1)) for s in sel]}")

first    : case of 9 chunks keeps chunk indices [0, 1, 2, 3]
last     : case of 9 chunks keeps chunk indices [5, 6, 7, 8]
head_tail: case of 9 chunks keeps chunk indices [0, 1, 7, 8]
uniform  : case of 9 chunks keeps chunk indices [0, 3, 5, 8]


## 6. The model: Phase-1 backbone + adaptation technique + hierarchical case head
* **full:** every backbone weight trainable. **partial:** only the top `train_top` layers (+ final norm). **unfreeze:** gradual unfreezing, one block of layers more each epoch from the top down (head only in epoch 1, everything in the last epoch) with discriminative learning rates (`lr_decay` per layer).
* **bitfit:** only added zero-init bias vectors. **adapter:** a bottleneck adapter (down, GELU, up with zero-init, residual) after every layer. **lorma:** multiplicative low-rank update (see the note at the top).
* **Knowledge distillation:** the student keeps the odd-numbered layers of the Phase-1 model and is fully fine-tuned with `(1-α)·CE(gold) + α·T²·KL(teacher‖student)` on case-level probabilities.
* **Head (all runs):** chunk vector -> linear + relative-position embedding -> attention pooling over chunks -> dropout -> linear to 2 classes.

In [61]:
class LoRMALinear(nn.Module):
    """Multiplicative low-rank adaptation (my reconstruction of LORMA; verify against reference [12]).
    side='out': y = (I + s B A)(W0 x)     side='in': y = W0((I + s B A) x)     B = 0 at init => identity update."""
    def __init__(self, base, r, alpha, dropout, side="auto"):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.side = side if side != "auto" else ("out" if base.out_features <= base.in_features else "in")
        d = base.out_features if self.side == "out" else base.in_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        if self.side == "out":
            y = self.base(x)
            return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale
        return self.base(x + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale)

class BiasLinear(nn.Module):
    """BitFit for a bias-free layer: add a zero-initialised bias vector and train only that."""
    def __init__(self, base):
        super().__init__()
        self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class Adapter(nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return self.up(F.gelu(self.down(x)))

LINEAR_TARGETS = {"in_proj", "out_proj", "q", "k", "v", "o", "gate", "up", "down"}
def inject(module, make):
    n = 0
    for name, child in list(module.named_children()):
        new = make(name, child) if isinstance(child, nn.Linear) else None
        if new is not None: setattr(module, name, new); n += 1
        else: n += inject(child, make)
    return n

def layer_index(name):
    m = re.search(r"(?:^|\.)(?:blocks|enc_blocks)\.(\d+)\.", name)
    return int(m.group(1)) if m else -1

class Phase4Model(nn.Module):
    def __init__(self, backbone, exp):
        super().__init__()
        self.backbone, self.method, self.exp = backbone, exp["method"], exp
        d = backbone_dim(backbone); self.adapters = None; self.n_wrapped = 0
        if self.method != "full":
            for p in backbone.parameters(): p.requires_grad = False
        L = len(layers(backbone))
        if self.method == "lorma":
            self.n_wrapped = inject(backbone, lambda n, c: LoRMALinear(c, exp["r"], exp["alpha"], exp["dropout"], exp.get("side", "auto")) if n in LINEAR_TARGETS else None)
        elif self.method == "bitfit":
            self.n_wrapped = inject(backbone, lambda n, c: BiasLinear(c) if n in LINEAR_TARGETS else None)
            for n, p in backbone.named_parameters():
                if n.endswith("bias"): p.requires_grad = True         # new bias vectors + existing bias-like parameters
        elif self.method == "adapter":
            self.adapters = nn.ModuleList([Adapter(d, exp["adapter_dim"]) for _ in range(L)])
        elif self.method == "partial":
            for i in range(L - exp["train_top"], L):
                for p in layers(backbone)[i].parameters(): p.requires_grad = True
            for p in final_norm(backbone).parameters(): p.requires_grad = True
        self.proj = nn.Linear(2 * d, HEAD_DIM); self.pos = nn.Embedding(8, HEAD_DIM)
        if AGG == "attn": self.att_in, self.att_out = nn.Linear(HEAD_DIM, HEAD_DIM), nn.Linear(HEAD_DIM, 1, bias=False)
        if AGG == "gru": self.gru = nn.GRU(HEAD_DIM, HEAD_DIM // 2, batch_first=True, bidirectional=True)
        self.drop, self.cls = nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_DIM, 2)
        if self.method == "unfreeze": self.set_unfreeze(0.0)

    def set_unfreeze(self, frac):
        """Gradual unfreezing: the top ceil(frac*L) layers (and the final norm) train; embeddings only when everything is unfrozen."""
        L = len(layers(self.backbone)); n_open = math.ceil(frac * L - 1e-9)
        for n, p in self.backbone.named_parameters():
            i = layer_index(n)
            if i >= 0: p.requires_grad = i >= L - n_open
            elif "norm" in n: p.requires_grad = frac > 0
            else: p.requires_grad = frac >= 1.0

    def hidden(self, b):
        m = self.backbone; x = embed(m)(b["ids"])
        return run_blocks(m, x, b["tok_mask"], self.adapters)

    def forward(self, b):
        h = self.hidden(b); N = h.size(0)
        v = b["tok_mask"].to(h.dtype)
        mean = (h * v[..., None]).sum(1) / v.sum(1, keepdim=True)
        last = h[torch.arange(N, device=h.device), b["lens"] - 1]
        cv = F.gelu(self.proj(torch.cat([mean, last], -1))) + self.pos(b["bucket"])
        Dn, C = b["nch"].numel(), int(b["nch_cpu"].max())
        x = cv.new_zeros(Dn, C, cv.size(-1)).index_put((b["owner"], b["rank"]), cv)
        cmask = torch.arange(C, device=h.device)[None] < b["nch"][:, None]
        w = None
        if AGG == "attn":
            s = self.att_out(torch.tanh(self.att_in(x))).squeeze(-1).masked_fill(~cmask, -1e4)
            w = torch.softmax(s.float(), -1).to(x.dtype); ctx = (w[..., None] * x).sum(1)
        elif AGG == "gru":
            pk = nn.utils.rnn.pack_padded_sequence(x, b["nch_cpu"], batch_first=True, enforce_sorted=False)
            out, _ = self.gru(pk); out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=C)
            ctx = (out * cmask[..., None]).sum(1) / cmask.sum(1, keepdim=True)
        else:
            ctx = (x * cmask[..., None]).sum(1) / cmask.sum(1, keepdim=True)
        return self.cls(self.drop(ctx)), w

def build_model():
    bb = build_backbone()
    if EXP.get("student_keep") == "odd": prune_layers(bb, list(range(1, len(layers(bb)), 2)))
    return Phase4Model(bb, EXP).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = build_model(); TOTAL_P, TRAIN_P = count_params(_m)
print(f"{EXPERIMENT}: total {TOTAL_P / 1e6:.2f}M params | trainable at start {TRAIN_P / 1e6:.3f}M ({100 * TRAIN_P / TOTAL_P:.2f}%) | {len(layers(_m.backbone))} backbone layers"
      + (f" | {_m.n_wrapped} linear layers adapted" if _m.n_wrapped else ""))
del _m

M4_lorma: total 112.93M params | trainable at start 1.896M (1.68%) | 20 backbone layers | 65 linear layers adapted


## 7. Metrics, evaluation and teacher loading
* Macro-F1 over the two classes (primary), macro precision/recall, accuracy. Reported at threshold 0.5 and at the dev-tuned threshold.
* `predict` returns, per case, the log-probabilities, the attention weights over its chunks and their relative positions: used later for the section analysis.
* For KD presets the teacher's case probabilities are loaded from `TEACHER_PKL`; the student trains on the training cases the teacher file contains.

In [62]:
def bin_scores(y, p):
    y, p = np.asarray(y), np.asarray(p); per = {}
    for c in (0, 1):
        tp = int(((y == c) & (p == c)).sum()); fp = int(((y != c) & (p == c)).sum()); fn = int(((y == c) & (p != c)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[c] = dict(p=pr, r=rc, f=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((y == c).sum()))
    return dict(macro_f1=(per[0]["f"] + per[1]["f"]) / 2, macro_p=(per[0]["p"] + per[1]["p"]) / 2, macro_r=(per[0]["r"] + per[1]["r"]) / 2,
                accuracy=float((y == p).mean()) if len(y) else 0.0, per_class=per)

def tune_threshold(y, p1):
    best = (0.5, bin_scores(y, (p1 >= 0.5).astype(int))["macro_f1"])
    for t in np.unique(np.clip(np.quantile(p1, np.linspace(0.02, 0.98, 97)), 0.02, 0.98)):
        f = bin_scores(y, (p1 >= t).astype(int))["macro_f1"]
        if f > best[1] + 1e-9: best = (float(t), f)
    return best

@torch.no_grad()
def predict(model, docs, policy=None, M=None):
    """-> log-probs (n,2), attention weights per case (list), relative chunk positions per case (list)."""
    model.eval(); lps, atts = [], []
    for i in range(0, len(docs), INFER_DOCS):
        chunk = docs[i:i + INFER_DOCS]; b = collate(chunk, policy, M)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            logits, w = model(b)
        lps.append(F.log_softmax(logits.float(), -1).cpu().numpy())
        if w is not None:
            w = w.float().cpu().numpy(); atts += [w[j, :int(b["nch_cpu"][j])] for j in range(len(chunk))]
    return np.concatenate(lps), atts

def evaluate(model, docs):
    t0 = time.time(); lp, atts = predict(model, docs); y = np.array([d["y"] for d in docs]); p1 = np.exp(lp[:, 1])
    sc = bin_scores(y, lp.argmax(1)); return sc, lp, atts, time.time() - t0

TEACHER = None
if EXP.get("kd_alpha") is not None:
    with open(TEACHER_PKL, "rb") as f: TEACHER = pickle.load(f)
    print(f"teacher: {TEACHER['meta']['experiment']} ({TEACHER['meta']['total_params'] / 1e6:.1f}M params, dev macro-F1 {TEACHER['meta']['dev_macro_f1']:.4f}), "
          f"{len(TEACHER['train'])} training cases with teacher outputs")

def kd_loss(logits, b, T):
    ls = F.log_softmax(logits.float() / T, -1); pt = F.softmax(b["teach"] / T, -1)
    return F.kl_div(ls, pt, reduction="batchmean") * T * T

## 8. Training machinery
* Each epoch draws `TRAIN_DOCS_PER_EPOCH` cases (all of them if `None`) without replacement; a micro-batch is `DOCS_PER_MICRO` cases (each up to `MAX_CHUNKS` chunk sequences); `GRAD_ACCUM` micro-batches per optimizer step.
* Class-balanced cross-entropy; fp16 autocast with GradScaler; linear warm-up then cosine decay; the best epoch on dev (macro-F1 at 0.5) is restored.
* `unfreeze` changes which parameters train at the start of every epoch and uses one learning rate per layer (`lr_decay` per layer below the top).

In [63]:
def class_weights(docs):
    c = np.bincount([d["y"] for d in docs], minlength=2).astype(np.float64)
    w = np.ones(2) if CLASS_WEIGHT == "none" else (c.sum() / (2 * np.maximum(c, 1)))
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_optimizer(model):
    exp = EXP; groups = []; L = len(layers(model.backbone))
    bb = [(n, p) for n, p in model.named_parameters() if n.startswith(("backbone.", "adapters."))]
    head = [(n, p) for n, p in model.named_parameters() if not n.startswith(("backbone.", "adapters."))]
    def add(lst, lr):
        dec = [p for n, p in lst if p.dim() >= 2]; nod = [p for n, p in lst if p.dim() < 2]
        for ps, wd in ((dec, WEIGHT_DECAY), (nod, 0.0)):
            if ps: groups.append({"params": ps, "lr": lr, "base_lr": lr, "weight_decay": wd})
    if exp["method"] == "unfreeze":
        by = defaultdict(list)
        for n, p in bb: by[layer_index(n)].append((n, p))
        for i, lst in by.items(): add(lst, exp["lr_backbone"] * exp["lr_decay"] ** ((L - 1 - i) if i >= 0 else L))
    else:
        add([(n, p) for n, p in bb if p.requires_grad or exp["method"] == "unfreeze"], exp["lr_backbone"])
    add(head, exp["lr_head"])
    return torch.optim.AdamW(groups, betas=(0.9, 0.98))

def snapshot(model):
    if EXP["method"] in ("full", "unfreeze", "partial"): return {n: v.detach().cpu().clone() for n, v in model.state_dict().items()}
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {n: v.detach().cpu().clone() for n, v in model.state_dict().items() if n in keep}

def run_training(train_docs, dev_docs, epochs, tag, use_kd, docs_per_epoch):
    model = build_model(); cw = class_weights(train_docs); opt = make_optimizer(model)
    scaler = torch.amp.GradScaler(device, enabled=device == "cuda")
    n_ep = min(len(train_docs), docs_per_epoch or len(train_docs)); per_step = DOCS_PER_MICRO * GRAD_ACCUM
    steps_per_epoch = max(1, n_ep // per_step); total = steps_per_epoch * epochs; warm = max(1, int(WARMUP_FRAC * total)); step = 0
    rng = random.Random(SEED); best = dict(f1=-1.0, epoch=0, state=None); hist = []; t_start = time.time()
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{tag}] {n_ep} cases/epoch x {epochs} epochs = {steps_per_epoch * epochs} optimizer steps of {per_step} cases")
    for ep in range(1, epochs + 1):
        if EXP["method"] == "unfreeze":
            model.set_unfreeze((ep - 1) / max(1, epochs - 1)); print(f"[{tag}] epoch {ep}: unfrozen fraction {(ep - 1) / max(1, epochs - 1):.2f}, trainable {count_params(model)[1] / 1e6:.2f}M")
        model.train(); order = rng.sample(train_docs, n_ep); run, nb = 0.0, 0; t_ep = time.time()
        for s in range(steps_per_epoch):
            f = (step + 1) / warm if step < warm else 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            for a in range(GRAD_ACCUM):
                j = (s * GRAD_ACCUM + a) * DOCS_PER_MICRO; b = collate(order[j:j + DOCS_PER_MICRO])
                with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                    logits, _ = model(b)
                loss = F.cross_entropy(logits.float(), b["y"], weight=cw)
                if use_kd: loss = (1 - EXP["kd_alpha"]) * loss + EXP["kd_alpha"] * kd_loss(logits, b, EXP["kd_temp"])
                if not math.isfinite(loss.item()): continue
                scaler.scale(loss / GRAD_ACCUM).backward(); run += loss.item(); nb += 1
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        rec = dict(epoch=ep, train_loss=run / max(nb, 1), sec=time.time() - t_ep)
        if dev_docs is not None:
            sc, _, _, sec_eval = evaluate(model, dev_docs); rec.update(dev_macro_f1=sc["macro_f1"], dev_acc=sc["accuracy"], eval_sec=sec_eval)
            if sc["macro_f1"] > best["f1"]: best = dict(f1=sc["macro_f1"], epoch=ep, state=snapshot(model))
            print(f"[{tag}] epoch {ep}/{epochs} | loss {rec['train_loss']:.4f} | dev macro-F1 {sc['macro_f1']:.4f} acc {sc['accuracy']:.3f} | train {rec['sec']:.0f}s + eval {sec_eval:.0f}s")
        else:
            print(f"[{tag}] epoch {ep}/{epochs} | loss {rec['train_loss']:.4f} | {rec['sec']:.0f}s")
        hist.append(rec)
    if best["state"] is not None: model.load_state_dict(best["state"], strict=False)
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    return model, dict(history=hist, best_epoch=best["epoch"] or len(hist), best_dev_f1=best["f1"], train_minutes=(time.time() - t_start) / 60,
                       peak_gpu_gb=peak, docs_per_epoch=n_ep)

### Throughput benchmark and budget fitting
Measures seconds per training case (forward + backward) and per dev case (forward only). If `EPOCHS` passes over every training case plus the dev evaluations would exceed `TARGET_TRAIN_MINUTES`, `TRAIN_DOCS_PER_EPOCH` is reduced so the whole schedule fits. Nothing is trained here, and tokenisation is done before timing.

In [64]:
use_kd = TEACHER is not None
train_pool = RAW["train"]
if use_kd:
    train_pool = [d for d in train_pool if d["key"] in TEACHER["train"]]
    assert train_pool, "the teacher file contains none of the training cases"
    for d in train_pool: d["teacher"] = TEACHER["train"][d["key"]].astype(np.float32)
    for d in RAW["dev"]: d["teacher"] = TEACHER["dev"][d["key"]].astype(np.float32)
    if not ALLOW_LARGER_STUDENT: assert TOTAL_P < TEACHER["meta"]["total_params"], \
        f"student ({TOTAL_P / 1e6:.1f}M) is not smaller than the teacher ({TEACHER['meta']['total_params'] / 1e6:.1f}M); pick a larger teacher or set ALLOW_LARGER_STUDENT=True"
dev_pool = RAW["dev"] if MAX_DEV_DOCS is None else RAW["dev"][:MAX_DEV_DOCS]

def benchmark(n=4):
    m = build_model(); m.train(); docs = random.Random(1).sample(train_pool, min(n, len(train_pool)))
    for d in docs: get_ids(d)
    dd = dev_pool[:n]
    for d in dd: get_ids(d)
    def once(fwd_only):
        t0 = time.time()
        for d in (dd if fwd_only else docs):
            b = collate([d])
            with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                if fwd_only:
                    with torch.no_grad(): m.eval(); m(b)
                else:
                    m.train(); logits, _ = m(b); F.cross_entropy(logits.float(), b["y"]).backward()
        if device == "cuda": torch.cuda.synchronize()
        return (time.time() - t0) / max(len(docs if not fwd_only else dd), 1)
    once(False); once(True)                                         # warm-up
    tr, ev = once(False), once(True); m.zero_grad(set_to_none=True); del m
    return tr, ev

t_train, t_eval = benchmark()
full = len(train_pool) * EPOCHS * t_train + len(dev_pool) * EPOCHS * t_eval
print(f"benchmark: {t_train:.2f}s per training case, {t_eval:.2f}s per dev case | all {len(train_pool)} training cases x {EPOCHS} epochs + dev evals = about {full / 60:.0f} min")
if AUTO_FIT_BUDGET and TRAIN_DOCS_PER_EPOCH is None and full / 60 > TARGET_TRAIN_MINUTES:
    budget = TARGET_TRAIN_MINUTES * 60 - len(dev_pool) * EPOCHS * t_eval
    TRAIN_DOCS_PER_EPOCH = max(GRAD_ACCUM * DOCS_PER_MICRO, int(budget / (EPOCHS * t_train)))
    print(f"-> over the {TARGET_TRAIN_MINUTES}-minute budget: sampling {TRAIN_DOCS_PER_EPOCH} training cases per epoch "
          f"({TRAIN_DOCS_PER_EPOCH * EPOCHS} case-visits in total, a fresh random subset each epoch)")

benchmark: 0.34s per training case, 0.17s per dev case | all 31677 training cases x 3 epochs + dev evals = about 549 min
-> over the 90-minute budget: sampling 4361 training cases per epoch (13083 case-visits in total, a fresh random subset each epoch)


## 9. Phase A: train on the training split, select the epoch on the dev split, tune the threshold

In [65]:
model, infoA = run_training(train_pool, dev_pool, EPOCHS, f"{EXPERIMENT}/A", use_kd, TRAIN_DOCS_PER_EPOCH)
scA, lpA, attA, sec_inf = evaluate(model, dev_pool)
yA = np.array([d["y"] for d in dev_pool]); p1A = np.exp(lpA[:, 1])
thr, f_thr = tune_threshold(yA, p1A)
sc_thr = bin_scores(yA, (p1A >= thr).astype(int))
THRESH = thr if USE_TUNED_THRESHOLD else 0.5
base = bin_scores(yA, np.full(len(yA), int(np.mean(yA) >= 0.5)))
print(f"\nbest epoch {infoA['best_epoch']} | DEV macro-F1 @0.5: {scA['macro_f1']:.4f} | tuned threshold {thr:.3f}: {f_thr:.4f} | majority-class baseline {base['macro_f1']:.4f}")
print(f"macro-P {scA['macro_p']:.4f} macro-R {scA['macro_r']:.4f} accuracy {scA['accuracy']:.4f} | dev inference {sec_inf:.0f}s for {len(dev_pool)} cases | threshold used for the submission: {THRESH:.3f}")
for c in (0, 1):
    r = scA["per_class"][c]; print(f"  class {c}: P {r['p']:.3f} R {r['r']:.3f} F1 {r['f']:.3f} support {r['support']}")

[M4_lorma/A] 4361 cases/epoch x 3 epochs = 816 optimizer steps of 16 cases
[M4_lorma/A] epoch 1/3 | loss 0.6967 | dev macro-F1 0.5552 acc 0.594 | train 1674s + eval 318s
[M4_lorma/A] epoch 2/3 | loss 0.6241 | dev macro-F1 0.5573 acc 0.605 | train 1636s + eval 311s
[M4_lorma/A] epoch 3/3 | loss 0.5817 | dev macro-F1 0.5413 acc 0.597 | train 1620s + eval 311s

best epoch 2 | DEV macro-F1 @0.5: 0.5573 | tuned threshold 0.339: 0.6408 | majority-class baseline 0.3333
macro-P 0.6828 macro-R 0.6046 accuracy 0.6046 | dev inference 311s for 1988 cases | threshold used for the submission: 0.339
  class 0: P 0.563 R 0.932 F1 0.702 support 994
  class 1: P 0.802 R 0.278 F1 0.413 support 994


### Which parts of a judgment matter? (section probe and attention)
The trained model is evaluated on the dev cases with **different parts of the document** as input: only the first chunk, only the last chunk, the first/last `MAX_CHUNKS`, head+tail, uniform. This is inference only, so it is cheap, but note the model was trained with `CHUNK_POLICY`, so a different policy is a small distribution shift. For attention pooling, the mean attention weight by relative position in the case is also reported.

In [66]:
probe_docs = dev_pool[:PROBE_MAX_DOCS]; yP = np.array([d["y"] for d in probe_docs]); probe = {}
for name, (pol, M) in {"first chunk only": ("first", 1), "last chunk only": ("last", 1), f"first {MAX_CHUNKS}": ("first", MAX_CHUNKS), f"last {MAX_CHUNKS}": ("last", MAX_CHUNKS),
                       f"head+tail {MAX_CHUNKS}": ("head_tail", MAX_CHUNKS), f"uniform {MAX_CHUNKS}": ("uniform", MAX_CHUNKS)}.items():
    lp, _ = predict(model, probe_docs, pol, M); probe[name] = bin_scores(yP, lp.argmax(1))["macro_f1"]
print(f"dev macro-F1 on {len(probe_docs)} cases by input section (trained with '{CHUNK_POLICY}', {MAX_CHUNKS} chunks):")
for k, v in probe.items(): print(f"   {k:18s} {v:.4f}")
att_by_pos = None
if AGG == "attn":
    acc = np.zeros(8); cnt = np.zeros(8)
    for d, w in zip(dev_pool, attA):
        sel, _ = select_chunks(len(get_ids(d)), CHUNK_POLICY, MAX_CHUNKS)
        for (st, en, rel), wi in zip(sel, w): bkt = min(int(rel * 8), 7); acc[bkt] += wi; cnt[bkt] += 1
    att_by_pos = (acc / np.maximum(cnt, 1)).tolist()
    print("mean attention weight by relative position in the case (octiles, 0 = start):", [f"{a:.2f}" for a in att_by_pos])

dev macro-F1 on 300 cases by input section (trained with 'head_tail', 4 chunks):
   first chunk only   0.3786
   last chunk only    0.5627
   first 4            0.4826
   last 4             0.5539
   head+tail 4        0.5478
   uniform 4          0.5539
mean attention weight by relative position in the case (octiles, 0 = start): ['0.20', '0.06', '0.02', '0.00', '0.03', '0.05', '0.02', '0.70']


### Error analysis
False positives vs false negatives, how errors depend on case length (number of 1024-token chunks) and model confidence, and the most confident mistakes.

In [67]:
pred = (p1A >= THRESH).astype(int); err = pred != yA
fp = int(((pred == 1) & (yA == 0)).sum()); fn = int(((pred == 0) & (yA == 1)).sum())
print(f"errors: {int(err.sum())} of {len(yA)} | false positives (predicted 1, gold 0): {fp} | false negatives (predicted 0, gold 1): {fn}")
ks = np.array([d.get("_k", 1) for d in dev_pool]); conf = np.maximum(p1A, 1 - p1A)
qs = np.quantile(ks, [1 / 3, 2 / 3]); bucket = np.digitize(ks, qs)
for b, nm in enumerate(["short cases", "medium cases", "long cases"]):
    m_ = bucket == b
    if m_.any(): print(f"   {nm:13s} (chunks {int(ks[m_].min())}-{int(ks[m_].max())}): n={int(m_.sum())}, error rate {err[m_].mean():.1%}")
print(f"mean confidence: correct {conf[~err].mean():.3f} | wrong {conf[err].mean() if err.any() else 0:.3f}")
worst = np.argsort(-(conf * err))[:5]
print("most confident mistakes (id, gold, P(1)):", [(dev_pool[i]["key"], int(yA[i]), round(float(p1A[i]), 3)) for i in worst if err[i]])

errors: 702 of 1988 | false positives (predicted 1, gold 0): 222 | false negatives (predicted 0, gold 1): 480
   short cases   (chunks 1-2): n=426, error rate 26.8%
   medium cases  (chunks 3-4): n=782, error rate 39.9%
   long cases    (chunks 5-52): n=780, error rate 35.4%
mean confidence: correct 0.751 | wrong 0.716
most confident mistakes (id, gold, P(1)): [('1960_229#2', 1, 0.004), ('1960_229', 1, 0.004), ('1975_57', 1, 0.004), ('1975_57#2', 1, 0.004), ('1966_191#2', 1, 0.028)]


## 10. Save teacher outputs, optional final retrain, predict the test set, write and validate the submission

In [68]:
def save_teacher(path, meta):
    out = {"meta": meta, "train": {}, "dev": {}, "test": {}}
    sub = random.Random(SEED).sample(RAW["train"], min(TEACHER_SAVE_TRAIN_DOCS, len(RAW["train"])))
    for name, docs in (("train", sub), ("dev", RAW["dev"]), ("test", RAW["test"])):
        lp, _ = predict(model, docs)
        for d, l in zip(docs, lp): out[name][d["key"]] = l.astype(np.float32)
    with open(path, "wb") as f: pickle.dump(out, f)

TEACHER_OUT = f"{EXP_DIR}/teacher_{EXPERIMENT}.pkl"
if SAVE_TEACHER and not use_kd:
    t0 = time.time()
    save_teacher(TEACHER_OUT, dict(experiment=EXPERIMENT, total_params=TOTAL_P, dev_macro_f1=scA["macro_f1"], note="Phase-A model"))
    print(f"saved teacher outputs -> {TEACHER_OUT} ({time.time() - t0:.0f}s). Use this path as TEACHER_PKL for a *_kd preset.")

if FINAL_RETRAIN:
    del model
    if device == "cuda": torch.cuda.empty_cache()
    model, infoB = run_training(train_pool + RAW["dev"], None, max(2, infoA["best_epoch"]), f"{EXPERIMENT}/B", use_kd, TRAIN_DOCS_PER_EPOCH)
else:
    infoB = dict(note="test predictions come from the phase-A model (best dev epoch)", best_epoch=infoA["best_epoch"])

t0 = time.time(); lpT, _ = predict(model, RAW["test"]); infer_sec = time.time() - t0
p1T = np.exp(lpT[:, 1]); predT = (p1T >= THRESH).astype(int)
submission = [{"id": d["id"], "pred": int(p)} for d, p in zip(RAW["test"], predT)]

# ---- validation of the file the scorer will read ----
assert len(submission) == len(RAW["test"]) and len({str(s["id"]) for s in submission}) == len(submission), "missing or duplicate cases"
assert all(isinstance(s["pred"], int) and s["pred"] in (0, 1) for s in submission), "pred must be the integer 0 or 1"
assert [s["id"] for s in submission] == [d["id"] for d in RAW["test"]], "ids differ from test.json"
print(f"test: {len(submission)} cases | predicted share of class 1: {predT.mean():.3f} (train {np.mean([d['y'] for d in RAW['train']]):.3f}) | threshold {THRESH:.3f} | inference {infer_sec:.0f}s")
print("sample:", submission[:3])
PRED_PATH = f"{EXP_DIR}/prediction.json"; ZIP_PATH = f"{EXP_DIR}/submission_{EXPERIMENT}.zip"
json.dump(submission, open(PRED_PATH, "w"))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH) as z: print("submission archive:", ZIP_PATH, "->", z.namelist())

saved teacher outputs -> /kaggle/working/phase4_cjpe/M4_lorma/teacher_M4_lorma.pkl (1080s). Use this path as TEACHER_PKL for a *_kd preset.
test: 1517 cases | predicted share of class 1: 0.399 (train 0.414) | threshold 0.339 | inference 239s
sample: [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}]
submission archive: /kaggle/working/phase4_cjpe/M4_lorma/submission_M4_lorma.zip -> ['prediction.json']


## 11. Save the experiment record and compare runs

In [69]:
result = dict(experiment=EXPERIMENT, description=EXP["desc"], method=EXP["method"], total_params=TOTAL_P, trainable_params_at_start=TRAIN_P,
              trainable_final=count_params(model)[1], backbone_layers=len(layers(model.backbone)),
              phaseA=dict(best_epoch=infoA["best_epoch"], dev_macro_f1_at_0p5=scA["macro_f1"], dev_macro_f1_tuned=f_thr, threshold=thr,
                          dev_macro_p=scA["macro_p"], dev_macro_r=scA["macro_r"], dev_accuracy=scA["accuracy"], per_class=scA["per_class"],
                          majority_baseline=base["macro_f1"], train_minutes=infoA["train_minutes"], peak_gpu_gb=infoA["peak_gpu_gb"],
                          docs_per_epoch=infoA["docs_per_epoch"], history=infoA["history"], section_probe=probe, attention_by_position=att_by_pos,
                          errors=dict(fp=fp, fn=fn)),
              phaseB=dict(infoB, test_inference_sec=infer_sec), threshold_used=THRESH,
              config=dict(chunk_tok=CHUNK_TOK, max_chunks=MAX_CHUNKS, policy=CHUNK_POLICY, agg=AGG, epochs=EPOCHS, class_weight=CLASS_WEIGHT,
                          lr_backbone=EXP["lr_backbone"], lr_head=EXP["lr_head"], kd=(EXP.get("kd_alpha"), EXP.get("kd_temp"))))
json.dump(result, open(f"{OUTPUT_DIR}/results_{EXPERIMENT}.json", "w"), indent=1, default=float)
rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/results_*.json")):
    r = json.load(open(p)); a = r["phaseA"]
    rows.append((r["experiment"], r["method"], r["total_params"] / 1e6, r["trainable_final"] / 1e6, a["dev_macro_f1_at_0p5"], a["dev_macro_f1_tuned"], a["train_minutes"], a["peak_gpu_gb"]))
print("\nexperiment  | method   | total M | trainable M (end) | dev F1 @0.5 | dev F1 tuned | train min | peak GB")
for r in rows: print(f"{r[0]:11s} | {r[1]:8s} | {r[2]:7.1f} | {r[3]:17.2f} | {r[4]:11.4f} | {r[5]:12.4f} | {r[6]:9.1f} | {r[7]:.1f}")
print("\nUpload", ZIP_PATH, "to Codabench (Phase 4), renamed to submission.zip.")


experiment  | method   | total M | trainable M (end) | dev F1 @0.5 | dev F1 tuned | train min | peak GB
M1_adapter  | adapter  |   113.5 |              2.44 |      0.5891 |       0.6760 |     110.5 | 8.2
M2_partial  | partial  |   111.5 |             24.05 |      0.6417 |       0.6496 |     101.3 | 5.9
M3_kd       | full     |    74.8 |             74.84 |      0.3333 |       0.4847 |      41.4 | 4.2
M4_lorma    | lorma    |   112.9 |              1.90 |      0.5573 |       0.6408 |      97.9 | 6.4

Upload /kaggle/working/phase4_cjpe/M4_lorma/submission_M4_lorma.zip to Codabench (Phase 4), renamed to submission.zip.


## Notes for the report and next steps
* All presets share the document representation, head, class weights, budget and decoding; differences come from the adaptation technique alone. Trainable parameters, training minutes, peak GPU memory, per-class scores, the section probe and attention by position are saved in `results_*.json`.
* Ablations that need no new technique (they do **not** count toward the six): `CHUNK_POLICY` (`first`/`last`/`head_tail`/`uniform`), `MAX_CHUNKS`, `AGG` (`attn`/`gru`/`mean`), `CLASS_WEIGHT="none"`, `USE_TUNED_THRESHOLD=False`.
* **Compute honesty:** if the benchmark cell had to sample cases per epoch, the models saw only part of the training data per epoch. State this in the report next to the scores, and keep the same budget for every preset so comparisons stay fair.
* The six submissions must involve different Phase-1 models and include **LORMA** and **knowledge distillation**. AdaLoRA (also on the Phase-4 list) is not implemented here.
* Threshold tuning uses the dev split that also selects the epoch, so the dev score at the tuned threshold is slightly optimistic; the Codabench score is the honest number.